# 06 — Recomendação Top-5 e Análise de Skills

Este notebook implementa e demonstra a **Issue 3 — Ranking Top-5 e análise de skills**
do projeto **JobMatch AI**.

O objetivo é, a partir de um **currículo em texto**, recomendar as vagas mais
aderentes ao perfil do candidato e analisar quais habilidades ele já possui e
quais ainda precisa desenvolver.

As funções ficam centralizadas em dois módulos reutilizáveis:

- `src/recommendation.py` — recomendação Top-N com **TF-IDF + similaridade por cosseno**;
- `src/skills.py` — comparação entre as **skills do candidato** e as **skills da vaga**.

A demonstração usa o arquivo de exemplo `data/sample/vagas_exemplo.csv`, mas o
mesmo código funciona com a base final `data/processed/vagas_enriquecidas_lite.parquet`,
pois ambos seguem o mesmo contrato de colunas (Issue 1).

## Decisões de implementação

**Recomendação (`recomendar_vagas`)**

1. O texto de cada vaga vem da coluna `texto_vaga_completo` (título + descrição + skills).
2. Aplicamos **TF-IDF** (unigramas e bigramas, com remoção de *stopwords* em inglês)
   sobre todas as vagas, gerando uma matriz esparsa.
3. O currículo é projetado no mesmo espaço vetorial.
4. Calculamos a **similaridade por cosseno** entre o currículo e cada vaga.
5. Ordenamos por similaridade e retornamos as **Top-5** vagas, com `score_aderencia`
   (0 a 1) e `aderencia_pct` (0 a 100).

A classe `RecomendadorVagas` ajusta o TF-IDF **uma única vez** e permite reaproveitar
a matriz em várias consultas — formato ideal para a interface Streamlit.

**Análise de skills (`comparar_skills`)**

1. As skills da vaga são normalizadas (sem acento, em minúsculas e sem duplicatas).
2. Cada skill é procurada no texto do currículo respeitando limites de palavra.
3. Retornamos as **skills compatíveis**, as **skills faltantes** e o
   **percentual de aderência** de skills, além de sugestões de desenvolvimento.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", 70)

print("Raiz do projeto:", PROJECT_ROOT)

Raiz do projeto: /home/user/jobmatch-ai


In [ ]:
from src.recommendation import (
    RecomendadorVagas,
    recomendar_vagas,
    carregar_base_vagas,
)
from src.skills import (
    comparar_skills,
    sugerir_desenvolvimento,
    extrair_vocabulario_skills,
    extrair_skills_do_curriculo,
    analisar_skills_recomendacoes,
)

print("Modulos de recomendacao e skills importados com sucesso.")

Modulos de recomendacao e skills importados com sucesso.


## 1. Carregar a base de vagas

A função `carregar_base_vagas` procura primeiro a base final em `data/processed/`
e, se ela ainda não existir, usa automaticamente o arquivo de exemplo
`data/sample/vagas_exemplo.csv`. Isso permite testar o módulo sem depender da
composição completa (Issue 1).

In [ ]:
df_vagas = carregar_base_vagas(project_root=PROJECT_ROOT)

print("Formato da base de vagas:", df_vagas.shape)
print("Colunas:", list(df_vagas.columns))

Formato da base de vagas: (50, 15)
Colunas: ['job_id', 'title', 'company_name', 'location', 'skills', 'texto_vaga_completo', 'description_preview', 'formatted_work_type', 'formatted_experience_level', 'remote_status', 'remote_allowed_flag', 'possui_salario', 'salary_range_text', 'pay_period', 'job_posting_url']


In [ ]:
colunas_visao = ["job_id", "title", "company_name", "location", "skills", "salary_range_text"]
df_vagas[colunas_visao].head()

,job_id,title,company_name,location,skills,salary_range_text
0,3902944011,Senior Automation Engineer - Power Systems,Current Power,"Houston, TX",engineering; information technology,Não informado
1,3901960222,DISH Installation Technician - Field,DISH Network,"Orange, TX",engineering; information technology,19.75 - 19.75 (HOURLY)
2,3900944095,Order Builder,"Coca-Cola Bottling Company UNITED, Inc.","Oxford, AL",management; manufacturing,Não informado
3,3903878594,"Mountain Multimedia Journalist, KMGH",Denver7 (KMGH-TV),"Denver, CO",marketing; public relations; writing/editing,Não informado
4,3905670593,Licensed Practical Nurse (LPN),BAYADA Home Health Care,"Teterboro, NJ",health care provider,30.00 - 35.00 (HOURLY)


## 2. Recomendação Top-5 por similaridade textual

Vamos usar como exemplo o currículo de um profissional de **gestão de projetos**.
A função `recomendar_vagas` recebe o texto do currículo e a base de vagas e
retorna o ranking ordenado por `score_aderencia`.

In [ ]:
curriculo_gestao = (
    "Profissional de gestao de projetos com 7 anos de experiencia. "
    "Atuei com HR project management, project management e stakeholder management. "
    "Tenho vivencia em change management, risk management e project documentation. "
    "Competencias de leadership, communication, teamwork e problem solving. "
    "Conduzi creating hr training programs e iniciativas de information technology aplicada a RH."
)

print(curriculo_gestao)

Profissional de gestao de projetos com 7 anos de experiencia. Atuei com HR project management, project management e stakeholder management. Tenho vivencia em change management, risk management e project documentation. Competencias de leadership, communication, teamwork e problem solving. Conduzi creating hr training programs e iniciativas de information technology aplicada a RH.


In [ ]:
recomendacoes = recomendar_vagas(curriculo_gestao, df_vagas, top_n=5)

colunas_rank = [
    "posicao",
    "score_aderencia",
    "aderencia_pct",
    "title",
    "company_name",
    "location",
    "salary_range_text",
]
recomendacoes[colunas_rank]

,posicao,score_aderencia,aderencia_pct,title,company_name,location,salary_range_text
0,1,0.3778,37.8,Human Resources Project Manager,"Sunrise Systems, Inc.","Boston, MA",Não informado
1,2,0.1082,10.8,Construction Consultant Internship 2024,Turner & Townsend,"Phoenix, AZ",20.00 - 25.00 (HOURLY)
2,3,0.0748,7.5,Laboratory Operations (LabOps) Project Manager,Planet Pharma,"San Mateo, CA",72.00 - 72.00 (HOURLY)
3,4,0.0706,7.1,Compliance Advisory Manager,Capital One,"Wilmington, DE",Não informado
4,5,0.0496,5.0,Human Resources Coordinator,Indotronix International Corporation,"New York, NY",Não informado


### Reaproveitando o vetorizador com `RecomendadorVagas`

Refazer o TF-IDF a cada consulta é desperdício quando a base é grande. A classe
`RecomendadorVagas` ajusta o vetorizador uma vez e responde a várias consultas
rapidamente — é assim que a interface deve usar o módulo.

In [ ]:
recomendador = RecomendadorVagas(df_vagas)

# A mesma matriz TF-IDF e reaproveitada em varias consultas.
consulta_rapida = recomendador.recomendar(curriculo_gestao, top_n=3)
consulta_rapida[["posicao", "aderencia_pct", "title", "company_name"]]

,posicao,aderencia_pct,title,company_name
0,1,37.8,Human Resources Project Manager,"Sunrise Systems, Inc."
1,2,10.8,Construction Consultant Internship 2024,Turner & Townsend
2,3,7.5,Laboratory Operations (LabOps) Project Manager,Planet Pharma


## 3. Análise de skills da vaga mais aderente

Agora comparamos as skills do currículo com as skills exigidas pela vaga melhor
ranqueada, usando `comparar_skills`.

In [ ]:
vaga_top1 = recomendacoes.iloc[0]

print("Vaga mais aderente:", vaga_top1["title"], "-", vaga_top1["company_name"])
print("Skills exigidas pela vaga:", vaga_top1["skills"])
print()

analise_skills = comparar_skills(curriculo_gestao, vaga_top1["skills"])

print("Aderencia de skills:", analise_skills["percentual_aderencia"], "%")
print()
print("Skills compativeis (", analise_skills["qtd_compativeis"], "):")
for skill in analise_skills["skills_compativeis"]:
    print("   +", skill)
print()
print("Skills faltantes (", analise_skills["qtd_faltantes"], "):")
for skill in analise_skills["skills_faltantes"]:
    print("   -", skill)

Vaga mais aderente: Human Resources Project Manager - Sunrise Systems, Inc.
Skills exigidas pela vaga: adaptability; change management; communication; confidentiality; creating hr communications; creating hr training programs; hr project management; information technology; leadership; managing complex employee events; pmp certification; problem solving; project documentation; project management; risk management; shrm-cp certification; smartsheets; stakeholder management; teamwork

Aderencia de skills: 63.2 %

Skills compativeis ( 12 ):
   + change management
   + communication
   + creating hr training programs
   + hr project management
   + information technology
   + leadership
   + problem solving
   + project documentation
   + project management
   + risk management
   + stakeholder management
   + teamwork

Skills faltantes ( 7 ):
   - adaptability
   - confidentiality
   - creating hr communications
   - managing complex employee events
   - pmp certification
   - shrm-cp certi

In [ ]:
print("Sugestoes de desenvolvimento:")
for sugestao in sugerir_desenvolvimento(analise_skills["skills_faltantes"]):
    print(" -", sugestao)

Sugestoes de desenvolvimento:
 - Desenvolver ou evidenciar a competência: adaptability.
 - Desenvolver ou evidenciar a competência: confidentiality.
 - Desenvolver ou evidenciar a competência: creating hr communications.
 - Desenvolver ou evidenciar a competência: managing complex employee events.
 - Desenvolver ou evidenciar a competência: pmp certification.
 - Há mais 2 skill(s) da vaga para desenvolver no futuro.


## 4. Skills do candidato a partir do vocabulário da base

Também é possível identificar as skills do candidato sem uma vaga específica,
comparando o currículo com **todas as skills observadas na base**. Isso ajuda a
interface a mostrar o "perfil de skills" do usuário.

In [ ]:
vocabulario_skills = extrair_vocabulario_skills(df_vagas)
print("Total de skills distintas na base:", len(vocabulario_skills))

skills_candidato = extrair_skills_do_curriculo(curriculo_gestao, vocabulario_skills)
print("Skills do candidato encontradas na base (", len(skills_candidato), "):")
print(skills_candidato)

Total de skills distintas na base: 58
Skills do candidato encontradas na base ( 13 ):
['change management', 'communication', 'creating hr training programs', 'hr project management', 'information technology', 'leadership', 'management', 'problem solving', 'project documentation', 'project management', 'risk management', 'stakeholder management', 'teamwork']


## 5. Top-5 com análise de skills integrada

A função `analisar_skills_recomendacoes` junta a recomendação Top-5 com a análise
de skills de cada vaga, produzindo exatamente o formato que a interface Streamlit
vai consumir: score de aderência textual + percentual de skills + compatíveis e
faltantes por vaga.

In [ ]:
def preparar_visao_skills(df):
    visao = df.copy()
    visao["skills_compativeis"] = visao["skills_compativeis"].apply(
        lambda s: "; ".join(s) if len(s) else "-"
    )
    visao["skills_faltantes"] = visao["skills_faltantes"].apply(
        lambda s: "; ".join(s) if len(s) else "-"
    )
    return visao


recomendacoes_skills = analisar_skills_recomendacoes(curriculo_gestao, recomendacoes)

colunas_final = [
    "posicao",
    "aderencia_pct",
    "title",
    "percentual_aderencia_skills",
    "skills_compativeis",
    "skills_faltantes",
]
preparar_visao_skills(recomendacoes_skills)[colunas_final]

,posicao,aderencia_pct,title,percentual_aderencia_skills,skills_compativeis,skills_faltantes
0,1,37.8,Human Resources Project Manager,63.2,change management; communication; creating hr training programs; h...,adaptability; confidentiality; creating hr communications; managin...
1,2,10.8,Construction Consultant Internship 2024,100.0,project management,-
2,3,7.5,Laboratory Operations (LabOps) Project Manager,100.0,information technology; project management,-
3,4,7.1,Compliance Advisory Manager,0.0,-,finance; sales
4,5,5.0,Human Resources Coordinator,4.8,communication,ability to thrive under pressure; advanced excel; analytical skill...


## 6. Segundo exemplo: currículo de tecnologia

Para mostrar que a solução é genérica, repetimos o fluxo com um currículo de
**engenharia de software**. Espera-se que vagas de tecnologia subam no ranking.

In [ ]:
curriculo_tech = (
    "Engenheiro de software com 6 anos de experiencia em Java, Spring Boot, "
    "microservices, REST APIs, Kubernetes e SQL Server. "
    "Trabalho com software engineering, design patterns e integration testing. "
    "Tenho atuacao em information technology e projetos de back-end."
)

recomendacoes_tech = recomendar_vagas(curriculo_tech, df_vagas, top_n=5)
recomendacoes_tech[["posicao", "aderencia_pct", "title", "company_name", "skills"]]

,posicao,aderencia_pct,title,company_name,skills
0,1,24.0,Java Architect,First Derivative,consulting; engineering; information technology
1,2,9.2,GoLang Developer (Capital One Experience Required) - ***W2 Only***,Convergenz,engineering
2,3,8.3,"AI Engineer, Biologics Engineering, Oncology R&D",AstraZeneca,engineering; information technology
3,4,4.8,Senior System Integration Tester,Swift Strategic Solutions Inc,information technology
4,5,3.5,Technical Support,Forrest T. Jones & Company,information technology


In [ ]:
vaga_tech_top1 = recomendacoes_tech.iloc[0]
analise_tech = comparar_skills(curriculo_tech, vaga_tech_top1["skills"])

print("Vaga mais aderente:", vaga_tech_top1["title"], "-", vaga_tech_top1["company_name"])
print("Aderencia de skills:", analise_tech["percentual_aderencia"], "%")
print("Skills compativeis:", analise_tech["skills_compativeis"])
print("Skills faltantes:", analise_tech["skills_faltantes"])

Vaga mais aderente: Java Architect - First Derivative
Aderencia de skills: 66.7 %
Skills compativeis: ['engineering', 'information technology']
Skills faltantes: ['consulting']


## 7. Conclusão

Este notebook entregou os itens previstos na Issue 3:

- **`recomendar_vagas`**: recebe um currículo e retorna as **Top-5 vagas** mais
  aderentes, ordenadas por `score_aderencia` (TF-IDF + similaridade por cosseno);
- **`comparar_skills`**: retorna **skills compatíveis**, **skills faltantes** e o
  **percentual de aderência** de skills;
- funções auxiliares (`sugerir_desenvolvimento`, `extrair_skills_do_curriculo`,
  `analisar_skills_recomendacoes`) que preparam a saída para a interface.

Os dois exemplos (gestão de projetos e tecnologia) mostraram rankings coerentes
com o perfil informado e uma análise de skills explicável.

**Observações sobre os dados:** no arquivo de exemplo, a coluna `skills` traz
categorias mais amplas do LinkedIn (ex.: *engineering*, *information technology*).
Quando a base final (Issue 1) com skills mais granulares estiver disponível,
basta apontar `carregar_base_vagas` para `data/processed/` — o código não muda.

**Próximo passo (Issue 4):** integrar `recomendar_vagas` e `comparar_skills` na
interface Streamlit, exibindo ranking, score, classificação Fit/No Fit, skills
compatíveis, skills faltantes e dados da vaga (empresa, local, salário e link).